In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("mast30034_project1")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

df = spark.read.parquet("../data/landing/yellow/")
df.printSchema()
print(f"Total rows, Jan-Jun 2025: {df.count():,}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/17 23:59:41 WARN Utils: Your hostname, DESKTOP-G75J20V, resolves to a loopback address: 127.0.1.1; using 172.25.167.45 instead (on interface eth0)
26/08/17 23:59:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/micha/mast30034-env/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/17 23:59:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



Total rows, Jan-Jun 2025: 24,083,384


In [2]:
zones = (
    spark.read
    .option("header", True)
    .csv("../data/landing/misc/taxi+_zone_lookup.csv")
)
zones.filter(F.col("Zone").contains("Airport")).show(truncate=False)

+----------+-------+-----------------+------------+
|LocationID|Borough|Zone             |service_zone|
+----------+-------+-----------------+------------+
|1         |EWR    |Newark Airport   |EWR         |
|132       |Queens |JFK Airport      |Airports    |
|138       |Queens |LaGuardia Airport|Airports    |
+----------+-------+-----------------+------------+



In [3]:
def log_shape(df, step: str, counts: list) -> None:
    """
    Record the row count after a preprocessing step.

    Args:
        df: DataFrame to count.
        step: Description of the step just applied.
        counts: Running list of (step, row_count) tuples.
    """
    n = df.count()
    counts.append((step, n))
    print(f"{step:<45} {n:>12,}")


shape_log = []
log_shape(df, "Raw yellow taxi, Jan-Jun 2025", shape_log)

Raw yellow taxi, Jan-Jun 2025                   24,083,384


In [4]:
# Viability check: is RatecodeID 2 (JFK flat fare) a substantial share of JFK pickups?
# If it's marginal, the flat-fare-vs-surge mechanism has little to analyse.
jfk = df.filter(F.col("PULocationID") == 132)
jfk_total = jfk.count()
print(f"Total JFK (LocationID 132) pickups, Jan-Jun 2025: {jfk_total:,}")

jfk.groupBy("RatecodeID").count().orderBy(F.desc("count")).show()

rc2_count = jfk.filter(F.col("RatecodeID") == 2).count()
print(f"RatecodeID 2 (JFK flat fare) share of JFK pickups: {rc2_count / jfk_total:.2%}")

Total JFK (LocationID 132) pickups, Jan-Jun 2025: 975,772


+----------+------+
|RatecodeID| count|
+----------+------+
|         2|478771|
|         1|423337|
|         5| 32028|
|         4| 30199|
|         3|  7222|
|      NULL|  4173|
|        99|    34|
|         6|     8|
+----------+------+



RatecodeID 2 (JFK flat fare) share of JFK pickups: 49.07%


## Validity filtering



In [ ]:
# Drop unknown zone codes. Both have Borough = "Unknown" in the
# zone lookup table (not real NYC taxi zones)
df = df.filter(
    ~F.col("PULocationID").isin(264, 265) & ~F.col("DOLocationID").isin(264, 265)
)
log_shape(df, "Drop unknown zone codes (264/265) in PU/DO", shape_log)

Drop unknown zone codes (264/265) in PU/DO      23,904,717


In [ ]:
# Drop RatecodeID 99 and true nulls. Per the yellow taxi data dictionary, 99 is the
# documented code for "Null/unknown" -- not a real rate code, an explicit placeholder
# for missing data. .
df = df.filter(F.col("RatecodeID").isin(1, 2, 3, 4, 5, 6))
log_shape(df, "Drop null/unknown RatecodeID (99, null; keep 1-6)", shape_log)

Drop null/unknown RatecodeID (99, null; keep 1-6)   18,233,691


In [7]:
# Drop non-positive fare_amount. fare_amount is the meter-calculated time-and-distance
# fare for a completed trip; values <= 0 represent voided trips, disputes, or refunds
# rather than a fare a driver actually earned.
df = df.filter(F.col("fare_amount") > 0)
log_shape(df, "Drop non-positive fare_amount", shape_log)

Drop non-positive fare_amount                   17,825,652


In [ ]:
# Drop non-positive trip_distance. A trip cannot have zero or negative distance
df = df.filter(F.col("trip_distance") > 0)
log_shape(df, "Drop non-positive trip_distance", shape_log)

Drop non-positive trip_distance                 17,635,957


In [ ]:
# Drop non-positive trip duration (dropoff <= pickup). Physically impossible for a
# completed trip
df = df.filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
log_shape(df, "Drop non-positive trip duration", shape_log)


[Stage 30:>                                                       (0 + 12) / 13]



Drop non-positive trip duration                 17,446,797


In [ ]:
# Restrict pickup datetime to the stated Jan-Jun 2025 window. TLC monthly releases are
# known to contain a small number of trips mis-stamped into adjacent months/years
df = df.filter(
    (F.col("tpep_pickup_datetime") >= "2025-01-01")
    & (F.col("tpep_pickup_datetime") < "2025-07-01")
)
log_shape(df, "Restrict pickup datetime to Jan-Jun 2025 window", shape_log)


[Stage 33:>                                                       (0 + 12) / 13]




[Stage 33:=================>                                       (4 + 9) / 13]



Restrict pickup datetime to Jan-Jun 2025 window   17,446,773


In [11]:
# Summary table for the report: rows remaining and rows dropped at each step.
import pandas as pd

shape_df = pd.DataFrame(shape_log, columns=["step", "rows_remaining"])
shape_df["rows_dropped"] = shape_df["rows_remaining"].diff().fillna(0).astype(int).abs()
shape_df

,step,rows_remaining,rows_dropped
0,"Raw yellow taxi, Jan-Jun 2025",24083384,0
1,Drop unknown zone codes (264/265) in PU/DO,23904717,178667
2,"Drop null/unknown RatecodeID (99, null; keep 1-6)",18233691,5671026
3,Drop non-positive fare_amount,17825652,408039
4,Drop non-positive trip_distance,17635957,189695
5,Drop non-positive trip duration,17446797,189160
6,Restrict pickup datetime to Jan-Jun 2025 window,17446773,24


## External dataset: BTS Airline On-Time Performance

Each monthly file covers all US domestic flights, so we read all six, filter to
flights arriving at JFK, and drop cancelled/diverted flights (they never land, so
they generate no curb pickups and have no meaningful arrival delay). This is a small
dataset relative to the taxi data, so `pandas` is used here rather than Spark.

In [12]:
import zipfile

import pandas as pd

# Only the columns relevant to joining flight arrivals against taxi pickup demand.
BTS_COLS = [
    "FlightDate", "Dest", "CRSArrTime", "ArrTime",
    "ArrDelayMinutes", "ArrDel15", "Cancelled", "Diverted",
]

bts_frames = []
for month in range(1, 7):
    zip_path = (
        "../data/landing/bts/"
        f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_{month}.zip"
    )
    with zipfile.ZipFile(zip_path) as z:
        csv_name = [name for name in z.namelist() if name.endswith(".csv")][0]
        with z.open(csv_name) as f:
            bts_frames.append(pd.read_csv(f, usecols=BTS_COLS, low_memory=False))

bts = pd.concat(bts_frames, ignore_index=True)
print(f"BTS flights, all US domestic, Jan-Jun 2025: {len(bts):,}")

BTS flights, all US domestic, Jan-Jun 2025: 3,446,676


In [ ]:
# Filter to flights arriving at JFK.
bts_jfk = bts[bts["Dest"] == "JFK"].copy()
print(f"BTS flights arriving at JFK, Jan-Jun 2025: {len(bts_jfk):,}")

# Drop cancelled flights: a flight that never lands at JFK generates no
# curb pickups, and Cancelled rows have no meaningful ArrDelayMinutes value.
bts_jfk = bts_jfk[(bts_jfk["Cancelled"] == 0) & (bts_jfk["Diverted"] == 0)]
print(f"BTS flights that actually landed at JFK: {len(bts_jfk):,}")

BTS flights arriving at JFK, Jan-Jun 2025: 51,395
BTS flights that actually landed at JFK: 50,586


## Save curated outputs

Later notebooks (`03_analysis`, `04_geospatial`, `05_modelling`) read from here.

In [14]:
df.write.mode("overwrite").parquet("../data/curated/yellow_filtered")
bts_jfk.to_parquet("../data/curated/bts_jfk_arrivals.parquet", index=False)
print("Saved curated yellow taxi and BTS JFK arrivals data.")


[Stage 36:=====================>                                   (5 + 8) / 13]



Saved curated yellow taxi and BTS JFK arrivals data.


## Preprocessing: HVFHV (rideshare comparison)

Same idea as the yellow taxi filtering above: keep the whole city, log shape at
every step, justify each filter against the FHVHV data dictionary. 

In [15]:
hvfhv = spark.read.parquet("../data/landing/hvfhv/")
hvfhv.printSchema()

hvfhv_shape_log = []
log_shape(hvfhv, "Raw HVFHV, Jan-Jun 2025", hvfhv_shape_log)

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [16]:
# Same justification as the yellow taxi zone filter: 264/265 have Borough = "Unknown"
# and are not real NYC taxi zones.
hvfhv = hvfhv.filter(
    ~F.col("PULocationID").isin(264, 265) & ~F.col("DOLocationID").isin(264, 265)
)
log_shape(hvfhv, "Drop unknown zone codes (264/265) in PU/DO", hvfhv_shape_log)

Drop unknown zone codes (264/265) in PU/DO     115,684,444



[Stage 41:===========================>                           (12 + 12) / 24]



In [ ]:
# Drop non-positive base_passenger_fare. Per the FHVHV data dictionary,
# base_passenger_fare is the base fare before tolls, tips, taxes, and fees
hvfhv = hvfhv.filter(F.col("base_passenger_fare") > 0)
log_shape(hvfhv, "Drop non-positive base_passenger_fare", hvfhv_shape_log)


[Stage 44:==>                                                     (1 + 12) / 24]



Drop non-positive base_passenger_fare          115,580,227



[Stage 44:=====================================>                  (16 + 8) / 24]



In [18]:
# Drop non-positive trip_miles. Same reasoning as trip_distance for yellow taxi:
# a trip cannot have zero or negative distance.
hvfhv = hvfhv.filter(F.col("trip_miles") > 0)
log_shape(hvfhv, "Drop non-positive trip_miles", hvfhv_shape_log)


[Stage 47:>                                                       (0 + 12) / 24]



Drop non-positive trip_miles                   115,565,413



[Stage 47:==============================================>         (20 + 4) / 24]



In [19]:
# Drop non-positive trip_time (seconds). Physically impossible for a completed trip,
# same reasoning as the pickup/dropoff duration filter for yellow taxi.
hvfhv = hvfhv.filter(F.col("trip_time") > 0)
log_shape(hvfhv, "Drop non-positive trip_time", hvfhv_shape_log)


[Stage 50:>                                                       (0 + 12) / 24]




[Stage 50:=========================>                             (11 + 12) / 24]



Drop non-positive trip_time                    115,565,413



[Stage 50:=====================================================>  (23 + 1) / 24]



In [20]:
# Restrict pickup_datetime to the stated Jan-Jun 2025 window, same reasoning as
# the yellow taxi date filter (mis-stamped records, single congestion-pricing regime).
hvfhv = hvfhv.filter(
    (F.col("pickup_datetime") >= "2025-01-01")
    & (F.col("pickup_datetime") < "2025-07-01")
)
log_shape(hvfhv, "Restrict pickup datetime to Jan-Jun 2025 window", hvfhv_shape_log)


[Stage 53:==>                                                     (1 + 12) / 24]




[Stage 53:======================>                                (10 + 12) / 24]



Restrict pickup datetime to Jan-Jun 2025 window  115,565,413



[Stage 53:=================================================>      (21 + 3) / 24]



In [ ]:
# Diagnostic: are shared/pooled rides common enough at JFK to need
# their own filter? 
hvfhv.filter(F.col("PULocationID") == 132).groupBy("shared_match_flag").count().show()


[Stage 56:==================>                                     (8 + 12) / 24]



+-----------------+-------+
|shared_match_flag|  count|
+-----------------+-------+
|                Y|   5002|
|                N|1683159|
+-----------------+-------+



In [22]:
# Summary table for the report, mirroring the yellow taxi shape_df above.
hvfhv_shape_df = pd.DataFrame(hvfhv_shape_log, columns=["step", "rows_remaining"])
hvfhv_shape_df["rows_dropped"] = (
    hvfhv_shape_df["rows_remaining"].diff().fillna(0).astype(int).abs()
)
hvfhv_shape_df

,step,rows_remaining,rows_dropped
0,"Raw HVFHV, Jan-Jun 2025",120995191,0
1,Drop unknown zone codes (264/265) in PU/DO,115684444,5310747
2,Drop non-positive base_passenger_fare,115580227,104217
3,Drop non-positive trip_miles,115565413,14814
4,Drop non-positive trip_time,115565413,0
5,Restrict pickup datetime to Jan-Jun 2025 window,115565413,0


In [23]:
# Viability check: does HVFHV have enough volume at JFK to be a meaningful
# comparison group against the yellow taxi flat fare?
hvfhv_jfk_total = hvfhv.filter(F.col("PULocationID") == 132).count()
print(f"Total HVFHV (rideshare) pickups at JFK, Jan-Jun 2025: {hvfhv_jfk_total:,}")


[Stage 59:======================>                                (10 + 12) / 24]




[Stage 59:================================>                      (14 + 10) / 24]



Total HVFHV (rideshare) pickups at JFK, Jan-Jun 2025: 1,688,161


In [24]:
hvfhv.write.mode("overwrite").parquet("../data/curated/hvfhv_filtered")
print("Saved curated HVFHV data.")


[Stage 62:==>                                                     (1 + 12) / 24]




[Stage 62:================>                                       (7 + 11) / 24]




[Stage 62:===================================>                    (15 + 9) / 24]



Saved curated HVFHV data.
